## P20_25

In [ ]:
import json, math, re
from pathlib import Path

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.models.resnet import ResNet, BasicBlock

RUNS = Path("data/resnet/hackin/aug_17_run")                       # <-- output dir of train_depth_sweep.py
DATA = Path("data/imagenet")     # <-- edit
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
MODELS = [m for m in MODEL_DEPTHS if (RUNS / m).exists()]

def color_of(name):
    """Plain nets: dark->light with depth. ResNet: red."""
    if name == "resnet74":
        return "crimson"
    depths = sorted(d for m, d in MODEL_DEPTHS.items() if m != "resnet74")
    return plt.cm.viridis(depths.index(MODEL_DEPTHS[name]) / (len(depths) - 1))

print(torch.__version__, device, "| runs found:", MODELS)

In [ ]:
# --- model definitions (mirrors train_depth_sweep.py) ---
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def blocks_of(model):
    return [b for s in (model.layer1, model.layer2, model.layer3, model.layer4)
            if isinstance(s, nn.Sequential) for b in s]

In [ ]:
# --- metrics loading: prefer metrics.npz, fall back to metrics.jsonl ---
def load_metrics(name):
    d = RUNS / name
    if (d / "metrics.npz").exists():
        return dict(np.load(d / "metrics.npz"))
    recs = [json.loads(l) for l in open(d / "metrics.jsonl") if l.strip()]
    out = {}
    for k in recs[0]:
        if k in ("block_grad_norms", "block_acts"):
            continue
        out[k] = np.array([r.get(k, np.nan) for r in recs], dtype=np.float64)
    out["block_grad_norms"] = np.array([r["block_grad_norms"] for r in recs])
    return out

M = {name: load_metrics(name) for name in MODELS}
for name in MODELS:
    print(f"{name:9s} records={len(M[name]['step']):5d} "
          f"last step={int(M[name]['step'][-1])}")

In [ ]:
def list_ckpts(name):
    return sorted(int(re.search(r"(\d+)", p.stem).group(1))
                  for p in (RUNS / name).glob("ckpt_step*.pt"))

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

print("plain74 checkpoints:", list_ckpts("plain74")[:5], "...",
      list_ckpts("plain74")[-2:])

## Accuracy Plots

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for m in MODELS:
    ax.plot(M[m]["step"], M[m]["val_top1"], color=color_of(m), label=m,
            lw=1.4, alpha=0.9)
ax.set(xlabel="step", ylabel="top-1 accuracy",
       title="quick val (5k subset), every 100 steps")
ax.legend(ncol=2, fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
for m in MODELS:
    steps = M[m]["step"]
    for ax, key, lbl in [(axes[0], "full_val_top1", "top-1"),
                          (axes[1], "full_val_top5", "top-5")]:
        v = M[m][key]
        keep = ~np.isnan(v)
        ax.plot(steps[keep], v[keep], color=color_of(m), label=m, lw=1.6)
for ax, lbl in zip(axes, ["top-1", "top-5"]):
    ax.set(xlabel="step", ylabel=f"{lbl} accuracy",
           title=f"full 50k val — {lbl}")
    ax.grid(alpha=0.3)
axes[0].legend(ncol=2, fontsize=9)
plt.tight_layout()

In [ ]:
M['plain8'].keys()

In [ ]:
M['plain8']['full_val_top1'][-1]

In [ ]:
M['plain8']['step'][-1]

In [ ]:
model = load_model('plain8')

In [ ]:
model

In [ ]:
model = load_model('plain14')

In [ ]:
model

In [ ]:
# for s,a in zip(M['plain14']['step'], M['plain14']['full_val_top1']):
#     print(s,a)

In [ ]:
# 40000.0 0.56742
# 50000.0 0.59748

In [ ]:
i=399

for m in ["plain8", "plain14", "plain20", "plain26", "plain34", "plain56", "plain74",  "resnet74"]:
    print(m, M[m]['step'][i], M[m]['full_val_top1'][i])

In [ ]:
i=399

for m in ['plain8', "plain14", "plain20", "plain26", "plain34", "plain56", "plain74",  "resnet74"]:
    print(m, M[m]['step'][-1], M[m]['full_val_top1'][-1])

In [ ]:
i=-1
M['plain20']['step'][i], M['plain20']['full_val_top1'][i]